# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Lane 1 + Lane 4 – *Explaining Search Performance Gaps Using Ranking Signals*

I chose to combine Lane 1 and Lane 4 because identifying pages with performance gaps is much more valuable when we can also explain why those gaps exist. My goal is to analyze ranking signals—such as content age, freshness, word count, content type, search intent, and other ranking-related signals available in the starter dataset content quality indicators—and connect them to pages that receive fewer clicks or perform below expectations. This approach focuses on finding actionable opportunities, helping turn search performance data into clear recommendations for improving rankings and traffic.


## 2. The question: decision, action, cost of a wrong call

My work addresses three connected questions:

1. **Where is the gap?**
   Which pages are underperforming relative to their ranking position?

2. **Why does the gap exist?**
   Which ranking signals — schema, readability, content structure —
   are missing or weak in those pages?

3. **What should we do about it?**
   For existing pages: which specific signals to fix, in what order,
   to close the gap?
   For new pages: which signals must be present from day one
   to avoid reproducing the same pattern?

**Who acts on it:**
Content writers and SEO strategists at FlyRank — both when
auditing existing pages for improvement AND when planning new ones.

**Cost of a wrong call:**
- Misidentify the cause → fix the wrong signal → gap persists,
  effort is wasted
- Miss a fixable page → underperforming page stays underperforming
  while competitors close the gap
- Wrong prevention advice → new pages launch with the same
  structural weaknesses → gap reproduces itself at scale

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [4]:
# Dataset size
print(f"Pages: {len(df):,}")
print(f"Clients: {df['client_id'].nunique()}")

# Pages with search visibility
print(f"Pages with impressions > 0: {(df['impressions_90d'] > 0).sum():,}")

# Average CTR
print(f"Average CTR: {df['ctr'].mean():.2%}")
print(df["trend_direction"].value_counts())

Pages: 30,000
Clients: 32
Pages with impressions > 0: 30,000
Average CTR: 51.07%
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


The starter dataset contains 30,000 pages from 32 clients. All pages have recorded search impressions, making them suitable for search performance analysis. In addition, 16,262 pages are currently trending downward, suggesting there are many opportunities to identify performance gaps and prioritize content improvements. This supports my choice of combining Lane 1 and Lane 4 because the dataset contains both search performance metrics (CTR, impressions, position) and content characteristics (such as content age, freshness, content type, and word count) that can help explain those gaps

## 4. Careful words: what I can and can't claim

**What this work CAN say:**
- "These pages show a measurable CTR gap at similar ranking positions"
- "Missing schema and low readability are consistently observed
   in underperforming pages — directional, not causal"
- "Fixing these specific signals is the most evidence-based
   next step available from this data"
- "New pages built with these signals present show fewer gaps
   in the observed dataset"

**What this work will NEVER claim:**
- ❌ "Fixing schema will increase CTR by X%"
  → We can show association, not predict magnitude
- ❌ "This page will rank better after the fix"
  → Rankings depend on factors outside this analysis
- ❌ "These are the only reasons for the gap"
  → Off-page factors (backlinks, brand, intent) are not modeled

The output is a prioritized, signal-backed action list —
honest about its limits, useful within them.